# Data for number of commits in main branch per author for all repositories of freeCodeCamp

In [ ]:
# all required libraries

import requests
import datetime
from dateutil.relativedelta import relativedelta
from dateutil.parser import parse
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import os
import calendar
import pandas as pd
import json
import numpy as np
import re
import shutil
import pprint
import time

In [ ]:
# classic tokens

secret_github_token_1 = 'ghp_aaa'
secret_github_token_2 = 'ghp_bbb'
secret_github_token_3 = 'ghp_ccc'
secret_github_token_4 = 'ghp_ddd'
secret_github_token_5 = 'ghp_eee'
secret_github_token_6 = 'ghp_fff'
secret_github_token_7 = 'ghp_ggg'

In [ ]:
# os.environ['GITHUB_TOKEN'] = secret_github_token
# GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
# HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

In [ ]:
owner = 'freeCodeCamp'

In [ ]:
# get last 6 months of data excluding current month
def get_date_range(n = 6):
    
    # today (yyyy-mm-dd)
    today = datetime.date.today()

    # date of first day of current month (yyyy-mm-dd)
    first_day_this_month = datetime.date(today.year, today.month, 1)

    # initialise array to contain tuples for past n months
    ranges = []

    # get each tuple
    for i in range(1, n + 1):

        # last day of iteration month (yyyy-mm-dd) (3rd in tuple)
        last_day = first_day_this_month - datetime.timedelta(days = 1)
        # convert to isoformat
        last_day_iso = last_day.isoformat() + 'T23:59:59Z'

        # first day of iteration month (yyyy-mm-01) (2nd in tuple)
        first_day = datetime.date(last_day.year, last_day.month, 1)
        # convert to isoformat
        first_day_iso = first_day.isoformat() + 'T00:00:00Z'

        # label: monthyear (mmmm yyyy) (1st in tuple)
        label = f'{first_day.strftime("%B %Y")}'

        # append tuples to ranges
        ranges.append((label, first_day_iso, last_day_iso))

        # set first_day_this_month to current iteration's first day to get the previous month in the next iteration
        first_day_this_month = first_day
    
    return(ranges)
# print(get_date_range())

In [ ]:
# creates text file in ascending chronological order of n monthyears preceding current month
# excludes current month

def output_past_six_months(n = 6, filename = 'past_six_months.txt'):
    if os.path.exists(filename):
        os.remove(filename)
    
    past_six_months = sorted(
        [i[0] for i in get_date_range(n)],
        key = lambda s: pd.to_datetime(s)
    )
    past_six_months_txt = ''
    for monthyear in past_six_months:
        past_six_months_txt += monthyear + '\n'

    with open(filename, 'w') as txt:
        txt.write(past_six_months_txt)

# output_past_six_months()

In [ ]:
# get list of repositories by username freeCodeCamp

def get_repo_list(username = owner, secret_github_token = secret_github_token_1):
    # username is the path parameter
    # secret_github_token is part of headers
    os.environ['GITHUB_TOKEN'] = secret_github_token
    GITHUB_TOKEN = os.environ['GITHUB_TOKEN']

    # constructing API call
    # 1st part: headers
    headers = {
        # 'who' this program is
        'Authorization': f'token {GITHUB_TOKEN}',
        # format wanted in program: json
        'Accept': 'application/vnd.github+json'
    }
    
    # full url address for api: base url + endpoint
    url = f'https://api.github.com/users/{username}/repos'
    
    # number of results per page
    per_page = 50
    # order by last updated (latest to earlies)
    sort = 'updated'
    # query parameters
    params = {
        'per_page': per_page,
        'sort': sort
    }
    
    # total no. of repos under username
    total_repos = 0
    # repo names under username
    all_repo_names = []
    
    while True:
        repos_resp = requests.get(url, headers = headers, params = params)
        repos_resp.raise_for_status()
        repos = repos_resp.json()
    
        total_repos += len(repos)
        for repo in repos:
            all_repo_names.append(repo['name'])
    
        # paginate using Link header
        nxt = repos_resp.links.get('next', {}).get('url')
        if not nxt:
            break
    
        url, params = nxt, None

    return(all_repo_names)

# x = get_repo_list()
# print(len(x))
# for i in sorted(x):
#     print(i)

In [ ]:
repo_names_check = sorted([re.sub(r'[^\w\s]', '', repo) for repo in get_repo_list()])
for i in repo_names_check:
    # print(i)
    pass

In [ ]:
print(len(set(repo_names_check)))

In [ ]:
# outputs text file of exact repository names as they are in github

def exact_names_to_txt(username = owner, secret_github_token = secret_github_token_1):
    output_txt = ''
    for exact_repo_name in sorted(get_repo_list(username = username, secret_github_token = secret_github_token)):
        output_txt += exact_repo_name + '\n'

    if os.path.exists('exact_repo_names.txt'):
        os.remove('exact_repo_names.txt')
    
    with open('exact_repo_names.txt', 'w') as txt:
        txt.write(output_txt)

In [ ]:
def to_iso_z(d: datetime.datetime) -> str:
    '''
    Format timezone rules-aware (meaning rules are known for
    - the UTC (coordinated universal time) offset
    - DST (daylight savings time) rules
    - a display name (tzname))
    datetime to UTC and return the datetime in the specified string format
    '''
    return(d.astimezone(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'))

def get_commits_of_one_repo(repo, foldername, owner = owner, n = 6, secret_github_token = secret_github_token_1):
    # repo = exact name of repo, each of which is in the output of get_repo_list()
    # this function is executed for each repo name in the output of get_repo_list()
    # owner and repo are path parameters
    # secret_github_token is part of headers

    os.environ['GITHUB_TOKEN'] = secret_github_token
    GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
    
    # constructing API call
    # 1st part: headers
    headers = {
        # 'who' this program is
        'Authorization': f'token {GITHUB_TOKEN}',
        # format wanted in program: json
        'Accept': 'application/vnd.github+json'
    }
    
    # full url address for api: base url + endpoint
    url = f'https://api.github.com/repos/{owner}/{repo}/commits'
    
    # query parameters
    per_page = 100
    earliest = datetime.datetime.fromisoformat(get_date_range(n)[-1][1])
    latest = datetime.datetime.fromisoformat(get_date_range(n)[0][2])
    
    params = {
        'per_page': per_page,
        'since': to_iso_z(earliest),
        'until': to_iso_z(latest)
    }
    
    # total commits
    total_commits = 0
    #array of all commits and their metadata for repo
    all_commits = []
    
    while True:
        commits_resp = requests.get(url, headers = headers, params = params)
        commits_resp.raise_for_status()
        page_items = commits_resp.json()
    
        for c in page_items:
            d = c['commit']['committer']['date'].replace('Z', '+00:00')
            commit_timestamp = datetime.datetime.fromisoformat(d)
            if earliest <= commit_timestamp <= latest:
                all_commits.append(c)
                total_commits += 1
    
        # paginate using the Link header
        nxt = commits_resp.links.get('next', {}).get('url')
        if not nxt:
            break
    
        url, params = nxt, None
    
    commits_dict = {
        'repo': [],
        'node_id': [], # commits[0]['node_id'],
        'commit_message': [], # commits[0]['commit']['message'],
        'author_name': [], # commits[0]['commit']['author']['name'],
        'author_email': [], # commits[0]['commit']['author']['email'],
        'author_timestamp': [], # commits[0]['commit']['author']['date'],
        'committer_name': [], # commits[0]['commit']['committer']['name'],
        'committer_email': [], # commits[0]['commit']['committer']['email'],
        'committer_timestamp': [], # commits[0]['commit']['committer']['date']
        'committer_monthyear': [],
        'committer_month': []
    }
    
    for c in all_commits:
        # get raw ISO 8601 timestamp string, e.g. '2025-08-26T17:40:56Z'
        iso = c['commit']['committer']['date']
        # parse once: make it timezone aware; 'Z' == UTC
        dt_utc = datetime.datetime.fromisoformat(iso.replace('Z', '+00:00'))
        
        commits_dict['repo'].append(repo)
        commits_dict['node_id'].append(c['node_id'])
        commits_dict['commit_message'].append(c['commit']['message']),
        commits_dict['author_name'].append(c['commit']['author']['name']),
        commits_dict['author_email'].append(c['commit']['author']['email']),
        commits_dict['author_timestamp'].append(c['commit']['author']['date']),
        commits_dict['committer_name'].append(c['commit']['committer']['name']),
        commits_dict['committer_email'].append(c['commit']['committer']['email']),
        commits_dict['committer_timestamp'].append(iso),
    
        # derived month fields
        commits_dict['committer_monthyear'].append(dt_utc.strftime('%B %Y'))
        commits_dict['committer_month'].append(pd.Timestamp(dt_utc).to_period('M'))
    
    commits_df = pd.DataFrame(commits_dict)
    
    # sort chronologically by month (ascending)
    commits_df = commits_df.sort_values('committer_month').reset_index(drop = True)
    
    # new folder for commit history files and each file name has the safe repo name (meaning no punctuation) prefix
    # './' refers to the current directory this jupyter notebook is in
    safe_repo_name = re.sub(r'[^\w\s]', '', repo)
    if len(safe_repo_name) == 0:
        folder_path, file_name = './' + foldername, '(blank)_commits_history.xlsx'
    else:
        folder_path, file_name = './' + foldername, f'{safe_repo_name}_commits_history.xlsx'

    # creates folder in current directory if it doesn't exist
    if not os.path.exists(folder_path):
        os.mkdir(folder_path)
    
    # concatenate folder and file to provide full path for saving excel file
    full_path = os.path.join(folder_path, file_name)

    # check if file already exists, in which case change the name
    suffix = 1
    while os.path.exists(full_path):
        full_path = os.path.join(folder_path, file_name[:-21] + '_' + str(suffix) + file_name[-21:])
        suffix += 1
    
    # print(len(commits_df))
    # commits_df.head()
    commits_df.to_excel(full_path, index = False)

In [ ]:
def create_all_commit_histories(n = 6, output_folder = 'commits_history', owner = owner, secret_github_token = secret_github_token_1):
    
    # delete output_folder and its contents if it exists already
    if os.path.exists('./' + output_folder):
        shutil.rmtree('./' + output_folder)
        
    # for each repository, get commit history
    for repo_name in get_repo_list(username = owner, secret_github_token = secret_github_token):
        get_commits_of_one_repo(
            repo = repo_name,
            foldername = output_folder,
            owner = owner,
            n = n,
            secret_github_token = secret_github_token
        )

In [ ]:
def run_query(
    query,
    graphql_endpoint = 'https://api.github.com/graphql',
    secret_github_token = secret_github_token_1,
    variables = None,
    max_retries = 5
):
    HEADERS = {
        'Authorization': f'Bearer {secret_github_token}'
    }

    for attempt in range(max_retries):
        try:
            request = requests.post(
                graphql_endpoint,
                json = {'query': query, 'variables': variables or {}},
                headers = HEADERS,
                timeout = 30
            )
            if request.status_code == 200:
                return request.json()
            if request.status_code in (500, 502, 503, 504):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
            raise Exception(f'Query failed with status code {request.status_code}: {request.text}')
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise Exception(f'Request failed after retries: {e}')

def adds_and_dels_df(
    repo_name,
    OWNER = owner,
    secret_github_token = secret_github_token_1,
    earliest = get_date_range()[-1][1],
    latest = get_date_range()[0][-1]
):
    after = None # starting value for pagination to get 100 records following after cursor

    graphql_endpoint = 'https://api.github.com/graphql'

    HEADERS = {
        'Authorization': f'Bearer {secret_github_token}'
    }

    query = """
        query($owner: String!, $repo_name: String!, $after: String, $earliest: GitTimestamp, $latest: GitTimestamp) {
            repository(owner: $owner name: $repo_name) {
                defaultBranchRef {
                    target {
                        ... on Commit {
                            history(first: 30 after: $after since: $earliest until: $latest) {
                                edges {
                                    cursor
                                }
                                nodes {
                                    id
                                    repository {
                                        name
                                    }
                                    additions
                                    deletions
                                    authoredDate
                                    committedDate
                                    author {
                                        email
                                        name
                                    }
                                    committer {
                                        email
                                        name
                                    }
                                }
                                pageInfo {
                                    endCursor
                                    hasNextPage
                                }
                            }
                        }
                    }
                }
            }
        }
    """

    variables = {
        'owner': OWNER,
        'repo_name': repo_name,
        'after': after,
        'earliest': earliest,
        'latest': latest
    }

    raw_last_hundred_results = run_query(
        query,
        graphql_endpoint = 'https://api.github.com/graphql',
        secret_github_token = secret_github_token,
        variables = variables
    )
    
    raw_results = [raw_last_hundred_results]
    
    endCursor = raw_last_hundred_results['data']['repository']['defaultBranchRef']['target']['history']['pageInfo']['endCursor']
    hasNextPage = raw_last_hundred_results['data']['repository']['defaultBranchRef']['target']['history']['pageInfo']['hasNextPage']
    
    while hasNextPage == True:
        variables['after'] = endCursor
        raw_temp_hundred_results = run_query(
            query,
            graphql_endpoint = graphql_endpoint,
            secret_github_token = secret_github_token,
            variables = variables
        )
        raw_results.append(raw_temp_hundred_results)
        endCursor = raw_temp_hundred_results['data']['repository']['defaultBranchRef']['target']['history']['pageInfo']['endCursor']
        hasNextPage = raw_temp_hundred_results['data']['repository']['defaultBranchRef']['target']['history']['pageInfo']['hasNextPage']
        if hasNextPage == False:
            break

    individual_commits = []
    
    for raw_result in raw_results:
        ind_c_arr = raw_result['data']['repository']['defaultBranchRef']['target']['history']['nodes']
        for ind_c in ind_c_arr:
            individual_commits.append(ind_c)
    
    additions_and_deletions_dict = {
        'repo': [commit['repository']['name'] for commit in individual_commits],
        'committer_timestamp': [commit['committedDate'] for commit in individual_commits],
        'author_timestamp': [commit['authoredDate'] for commit in individual_commits],
        'author_email': [commit['author']['email'] for commit in individual_commits],
        'committer_email': [commit['committer']['email'] for commit in individual_commits],
        'additions': [commit['additions'] for commit in individual_commits],
        'deletions': [-1*commit['deletions'] for commit in individual_commits]
    }

    temp_committer_timestamp_arr = [datetime.datetime.fromisoformat(i.replace('Z', '+00:00')) for i in additions_and_deletions_dict['committer_timestamp']]
    
    additions_and_deletions_dict['committer_monthyear'] = [date_time.strftime('%B %Y') for date_time in temp_committer_timestamp_arr]
    additions_and_deletions_dict['committer_month'] = [pd.Timestamp(date_time).to_period('M') for date_time in temp_committer_timestamp_arr]
    additions_and_deletions_dict['committer_date'] = [date_time.date() for date_time in temp_committer_timestamp_arr]
    
    additions_and_deletions_df = pd.DataFrame(additions_and_deletions_dict)

    additions_and_deletions_df = additions_and_deletions_df.sort_values('committer_timestamp').reset_index(drop = True)

    return(additions_and_deletions_df)

In [ ]:
def adds_and_dels_to_excel(
    repo_name,
    OWNER = owner,
    secret_github_token = secret_github_token_1,
    earliest = get_date_range()[-1][1],
    latest = get_date_range()[0][-1]
):
    params = {
        'OWNER': OWNER,
        'secret_github_token': secret_github_token,
        'repo_name': repo_name,
        'earliest': earliest,
        'latest': latest
    }

    # executes previous function to create the dataframe
    repo_additions_and_deletions_df = adds_and_dels_df(**params)
    
    foldername = 'additions_and_deletions'
    
    safe_repo_name = re.sub(r'[^\w\s]', '', repo_name)
    if len(safe_repo_name) == 0:
        folder_path, file_name = './' + foldername, '(blank)_additions_and_deletions.xlsx'
    else:
        folder_path, file_name = './' + foldername, f'{safe_repo_name}_additions_and_deletions.xlsx'
    
    # creates folder in current directory if it doesn't exist
    if not os.path.exists(folder_path):
        os.mkdir(folder_path)
    
    # concatenate folder and file to provide full paths for saving excel file
    full_path = os.path.join(folder_path, file_name)
    
    # check if file already exists, in which case change the name
    suffix = 1
    while os.path.exists(full_path):
        full_path = os.path.join(folder_path, file_name[:-len('_additions_and_deletions.xlsx')] + '_' + str(suffix) + file_name[-len('_additions_and_deletions.xlsx'):])
        suffix += 1
    
    repo_additions_and_deletions_df.to_excel(full_path, index = False)

In [ ]:
def create_all_adds_and_dels(
    output_folder = 'additions_and_deletions',
    OWNER = owner,
    secret_github_token = secret_github_token_1,
    earliest = get_date_range()[-1][1],
    latest = get_date_range()[0][-1]
):
    # delete output_folder and its contents if it exists already
    if os.path.exists('./' + output_folder):
        shutil.rmtree('./' + output_folder)

    failed_repos = []
    
    for repo in get_repo_list(username = OWNER, secret_github_token = secret_github_token):
        try:
            adds_and_dels_to_excel(
                repo_name = repo,
                OWNER = OWNER,
                secret_github_token = secret_github_token,
                earliest = earliest,
                latest = latest
            )
            print(f'Saved: {repo}')
        except Exception as e:
            print(f'Failed: {repo} -> {e}')
            failed_repos.append((repo, str(e)))

    return(failed_repos)

In [ ]:
def get_prs_of_one_repo(repo, owner, secret_github_token = secret_github_token_6):

    os.environ['GITHUB_TOKEN'] = secret_github_token
    GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
    
    # constructing API call
    # 1st part: headers
    headers = {
        # 'who' this program is
        'Authorization': f'token {GITHUB_TOKEN}',
        # format wanted in program: json
        'Accept': 'application/vnd.github+json'
    }
    
    # full url address for api: base url + endpoint
    # repo = 'infra'
    url = f'https://api.github.com/repos/{owner}/{repo}/pulls'
    
    earliest = datetime.datetime.fromisoformat(get_date_range(n = 29)[-1][1])
    latest = datetime.datetime.fromisoformat(get_date_range(n = 29)[0][2])
    
    params = {
        'state': 'all',
        'per_page': 100,
        'sort': 'updated',
        'direction': 'desc'
    }
    
    all_pulls = []
    stop_repo = False
    page_count = 0
    
    while True:
        pulls_resp = requests.get(url, headers = headers, params = params, timeout = 30)
        pulls_resp.raise_for_status()
        page_items = pulls_resp.json()
        page_count += 1

        if not page_items:
            break
    
        for p in page_items:
            updated_at_timestamp = datetime.datetime.fromisoformat(p['updated_at'].replace('Z', '+00:00'))

            if updated_at_timestamp < earliest:
                stop_repo = True
                break
    
            created_at_timestamp = datetime.datetime.fromisoformat(p['created_at'].replace('Z', '+00:00'))
            closed_at_timestamp = (datetime.datetime.fromisoformat(p['closed_at'].replace('Z', '+00:00')) if p['closed_at'] is not None else None)
            merged_at_timestamp = (datetime.datetime.fromisoformat(p['merged_at'].replace('Z', '+00:00')) if p['merged_at'] is not None else None)
            
            # created during this window
            is_created_in_range = earliest <= created_at_timestamp <= latest
            # merged during this window
            is_merged_in_range = merged_at_timestamp is not None and earliest <= merged_at_timestamp <= latest
            # closed during this window
            is_closed_in_range = closed_at_timestamp is not None and earliest <= closed_at_timestamp <= latest
            
            if is_created_in_range or is_merged_in_range or is_closed_in_range:
                all_pulls.append(p)
    
        if stop_repo:
            break
        
        # paginate using the Link header
        nxt = pulls_resp.links.get('next', {}).get('url')
        if not nxt:
            break
    
        url, params = nxt, None
    
    # dictionary (to become dataframe) for final dataframe
    pr_dict = {
        'pr_target_branch_repo_name': [],
        'pr_number': [],
        'pr_node_id': [],
        'pr_state': [],
        'pr_locked': [],
        'pr_title': [],
        'pr_user_login': [],
        'pr_user_id': [],
        'pr_user_node_id': [],
        'pr_created_at': [],
        'pr_updated_at': [],
        'pr_closed_at': [],
        'pr_merged_at': [],
        'pr_created_at_monthyear': [],
        'pr_updated_at_monthyear': [],
        'pr_closed_at_monthyear': [],
        'pr_merged_at_monthyear': [],
        'pr_created_at_month': [],
        'pr_updated_at_month': [],
        'pr_closed_at_month': [],
        'pr_merged_at_month': [],
        'pr_draft': [],
        'pr_source_branch_label': [],
        'pr_source_branch_ref': [],
        'pr_source_branch_latest_commit_sha': [],
        'pr_source_branch_user_login': [],
        'pr_source_branch_user_id': [],
        'pr_source_branch_user_node_id': [],
        'pr_source_branch_repo_full_name': [],
        'pr_source_branch_repo_name': [],
        'pr_target_branch_label': [],
        'pr_target_branch_commit_sha': [],
        'pr_target_branch_user_login': [],
        'pr_target_branch_user_id': [],
        'pr_target_branch_user_node_id': [],
        'pr_author_association': []
    }
    
    # fill out dictionary
    for p in all_pulls:
        created_at_timestamp = datetime.datetime.fromisoformat(p['created_at'].replace('Z', '+00:00')).replace(tzinfo=None)
        updated_at_timestamp = datetime.datetime.fromisoformat(p['updated_at'].replace('Z', '+00:00')).replace(tzinfo=None)
        closed_at_timestamp = (datetime.datetime.fromisoformat(p['closed_at'].replace('Z', '+00:00')).replace(tzinfo=None) if p['closed_at'] is not None else None)
        merged_at_timestamp = (datetime.datetime.fromisoformat(p['merged_at'].replace('Z', '+00:00')).replace(tzinfo=None) if p['merged_at'] is not None else None)

        head = p.get('head') or {}
        base = p.get('base') or {}

        head_user = head.get('user') or {}
        head_repo = head.get('repo') or {}
        base_user = base.get('user') or {}
        base_repo = base.get('repo') or {}
        pr_user = p.get('user') or {}
        
        pr_dict['pr_target_branch_repo_name'].append(base_repo.get('name'))
        pr_dict['pr_number'].append(p['number'])
        pr_dict['pr_node_id'].append(p['node_id'])
        pr_dict['pr_state'].append(p['state'])
        pr_dict['pr_locked'].append(p['locked'])
        pr_dict['pr_title'].append(p['title'])
        pr_dict['pr_user_login'].append(pr_user.get('login'))
        pr_dict['pr_user_id'].append(pr_user.get('id'))
        pr_dict['pr_user_node_id'].append(pr_user.get('node_id'))
        pr_dict['pr_created_at'].append(created_at_timestamp)
        pr_dict['pr_updated_at'].append(updated_at_timestamp)
        pr_dict['pr_closed_at'].append(closed_at_timestamp)
        pr_dict['pr_merged_at'].append(merged_at_timestamp)
        pr_dict['pr_draft'].append(p['draft'])
        pr_dict['pr_source_branch_label'].append(head.get('label'))
        pr_dict['pr_source_branch_ref'].append(head.get('ref'))
        pr_dict['pr_source_branch_latest_commit_sha'].append(head.get('sha'))
        pr_dict['pr_source_branch_user_login'].append(head_user.get('login'))
        pr_dict['pr_source_branch_user_id'].append(head_user.get('id'))
        pr_dict['pr_source_branch_user_node_id'].append(head_user.get('node_id'))
        pr_dict['pr_source_branch_repo_full_name'].append(head_repo.get('full_name'))
        pr_dict['pr_source_branch_repo_name'].append(head_repo.get('name'))
        pr_dict['pr_target_branch_label'].append(base.get('label'))
        pr_dict['pr_target_branch_commit_sha'].append(base.get('sha'))
        pr_dict['pr_target_branch_user_login'].append(base_user.get('login'))
        pr_dict['pr_target_branch_user_id'].append(base_user.get('id'))
        pr_dict['pr_target_branch_user_node_id'].append(base_user.get('node_id'))
        pr_dict['pr_author_association'].append(p['author_association'])
    
        pr_dict['pr_created_at_monthyear'].append(created_at_timestamp.strftime('%B %Y'))
        pr_dict['pr_updated_at_monthyear'].append(updated_at_timestamp.strftime('%B %Y'))
        pr_dict['pr_closed_at_monthyear'].append(closed_at_timestamp.strftime('%B %Y') if closed_at_timestamp is not None else None)
        pr_dict['pr_merged_at_monthyear'].append(merged_at_timestamp.strftime('%B %Y') if merged_at_timestamp is not None else None)
        
        pr_dict['pr_created_at_month'].append(pd.Timestamp(created_at_timestamp).to_period('M'))
        pr_dict['pr_updated_at_month'].append(pd.Timestamp(updated_at_timestamp).to_period('M'))
        pr_dict['pr_closed_at_month'].append(pd.Timestamp(closed_at_timestamp).to_period('M') if closed_at_timestamp is not None else None)
        pr_dict['pr_merged_at_month'].append(pd.Timestamp(merged_at_timestamp).to_period('M') if merged_at_timestamp is not None else None)
    
    # convert dictionary to dataframe
    pr_df = pd.DataFrame(pr_dict)
    
    return(pr_df)

In [ ]:
def one_pr_to_excel(
    repo,
    owner,
    secret_github_token = secret_github_token_6
):
    # executes previous function to create the dataframe
    repo_prs_df = get_prs_of_one_repo(repo, owner, secret_github_token)

    foldername = 'pull_requests'

    safe_repo_name = re.sub(r'[^\w\s]', '', repo)
    if len(safe_repo_name) == 0:
        folder_path, file_name = './' + foldername, '(blank)_prs.xlsx'
    else:
        folder_path, file_name = './' + foldername, f'{safe_repo_name}_prs.xlsx'

    # creates folder in current directory if it doesn't exist
    if not os.path.exists(folder_path):
        os.mkdir(folder_path)

    # concatenate folder and file to provide full paths for saving excel file
    full_path = os.path.join(folder_path, file_name)

    # check if file already exists, in which case change the name
    suffix = 1
    while os.path.exists(full_path):
        full_path = os.path.join(folder_path, file_name[:-len('_prs.xlsx')] + '_' + str(suffix) + file_name[-len('_prs.xlsx'):])
        suffix += 1

    # convert datetime columns to timezone-unaware datetimes
    datetime_cols = [
        'pr_created_at',
        'pr_updated_at',
        'pr_closed_at',
        'pr_merged_at'
    ]

    for col in datetime_cols:
        if col in repo_prs_df.columns:
            repo_prs_df[col] = pd.to_datetime(repo_prs_df[col], errors='coerce').dt.tz_localize(None)

    repo_prs_df.to_excel(full_path, index = False)

In [ ]:
def create_all_prs(
    output_folder = 'pull_requests',
    OWNER = owner,
    secret_github_token = secret_github_token_6
):
    # delete output_folder and its contents if it exists already
    if os.path.exists('./' + output_folder):
        shutil.rmtree('./' + output_folder)

    os.mkdir('./' + output_folder)

    failed_repos = []
    
    for repo in get_repo_list(username = OWNER, secret_github_token = secret_github_token_7):
        try:
            one_pr_to_excel(
                repo = repo,
                owner = OWNER,
                secret_github_token = secret_github_token
            )
            print(f'Saved: {repo}')
        except Exception as e:
            print(f'Failed: {repo} -> {e}')
            failed_repos.append((repo, str(e)))

    return(failed_repos)

In [ ]:
# # create all repos' commits history excel files for past 6 months excluding current month
# create_all_commit_histories(secret_github_token = secret_github_token_1)

# # outputs past 6 months excluding current month in chronological order
# output_past_six_months()

# # writes repo names under owner to text file
# exact_names_to_txt(username = owner, secret_github_token = secret_github_token_2)

# # for commit histories for past 29 months excluding current month
# create_all_commit_histories(n = 29, output_folder = 'for_12_kpi', owner = owner, secret_github_token = secret_github_token_3)

# # outputs past 29 months excluding current month in chronological order
# output_past_six_months(n = 29, filename = 'past_29_months.txt')

# # outputs all excel files of additions and deletions for each repo
# create_all_adds_and_dels(secret_github_token = secret_github_token_4)

# # create history of all prs for the past 29 months excluding the current month
# create_all_prs(secret_github_token = secret_github_token_6)

# Read in data from excel for checks - NOT part of data transformation pipeline

In [ ]:
foldername = 'pull_requests'

arr_pr_history_dfs = []
count = 0

for filename in sorted(os.listdir('./' + foldername)):
    temp_df = pd.read_excel(foldername + '/' + filename)
    count += 1
    arr_pr_history_dfs.append(temp_df)

# print(count)

all_pr_history_dfs = pd.concat(arr_pr_history_dfs)

all_pr_history_dfs.head()

In [ ]:
repo_list_file, past_six_months_file = 'exact_repo_names.txt', 'past_six_months.txt'

with open(repo_list_file, 'r') as repo_txt_file:
    repos_arr = [line.rstrip() for line in repo_txt_file]

with open(past_six_months_file, 'r') as past_six_months_txt_file:
    monthyears_sorted = [line.rstrip() for line in past_six_months_txt_file]

selected_repos = repos_arr
selected_month = monthyears_sorted[-1]

print(len(selected_repos))
print(selected_month)

In [ ]:
def read_xlsx(filename):
    df = pd.read_excel('commits_history/' + filename)
    return(df)

arr_dfs = []
for i in sorted(os.listdir('./commits_history')):
    arr_dfs.append(read_xlsx(i))

commits_df = pd.concat(arr_dfs)
print(commits_df)

In [ ]:
df_for_pie = (
    commits_df[['repo', 'committer_month', 'author_email', 'node_id']]
        .groupby(['repo', 'committer_month', 'author_email'])
        .count()
        .rename(columns = {'node_id': 'count of commits'})
        .reset_index()
        .sort_values('committer_month')
)
df_for_pie['committer_monthyear'] = (
    pd.to_datetime(df_for_pie['committer_month'] + '-01')
        .dt.strftime('%B %Y')
)

print(df_for_pie)

In [ ]:
def exact_names_to_txt():
    output_txt = ''
    for exact_repo_name in sorted(get_repo_list()):
        output_txt += exact_repo_name + '\n'
    
    with open('exact_repo_names.txt', 'w') as txt:
        txt.write(output_txt)

In [ ]:
with open('exact_repo_names.txt', 'r') as txt_file:
    exact_repo_names = [line.rstrip() for line in txt_file]

print(exact_repo_names)

In [ ]:
# KPIs for contributor commits
# Total number of commits in the past 12 months excluding the current selected one with delta for YoY (so a total of 24 months of data needed)
# Total number of commits in the past month with delta for MoM
# Total number of commits so far in the quarter excluding current month with delta for QoQ

past_29_months = get_date_range(29)
count = 0
for i in past_29_months:
    print(count, i)
    count += 1
# print(past_24_months[12:]) # prev
# print(past_24_months[:12]) # original

In [ ]:
# provides all the data required for the 12-month KPI
# create_all_commit_histories(n = 29, output_folder = 'for_12_kpi', owner = owner)

In [ ]:
# sum node_ids grouped by monthyears
# concatenate all excel files in the folder
# do with read_in_and_concat_commits_history_dfs('for_12_kpi')
# filter to only include previous 12 months excluding selected month based on selection

def grouping_and_aggregation_for_12_kpi(ungrouped_df, selected_repos, selected_month):

    mask = ungrouped_df['repo'].isin(selected_repos) if selected_repos else True
    # mask = mask & ungrouped_df['committer_monthyear'].isin(



In [ ]:
def read_in_and_concat_commits_history_dfs(foldername = 'commits_history'):

    '''
    reads in all commits history excel files and unionises them, outputting the concatenated table that is completely unaggregated
    '''
    
    # initialise empty array where each element will be one dataframe
    arr_commits_history_dfs = []

    # populate array
    for filename in sorted(os.listdir('./' + foldername)):
        temp_df = pd.read_excel(foldername + '/' + filename)
        arr_commits_history_dfs.append(temp_df)

    # concatenate (i.e., union all) dataframes in the same order the files appear in the directory
    all_commits_history_dfs = pd.concat(arr_commits_history_dfs)

    # output concatenated dataframe
    return(all_commits_history_dfs)

for_12_kpi_dfs = read_in_and_concat_commits_history_dfs(foldername = 'for_12_kpi')

# for_12_kpi_dfs.head(10)

In [ ]:
# committer_monthyear = for_12_kpi_dfs['committer_monthyear'].unique()
# print(committer_monthyear)
# print(type(committer_monthyear))
# committer_num_month = pd.to_datetime(pd.Series(sorted(committer_monthyear, key = lambda s: pd.to_datetime(s)))).dt.strftime('%Y-%m')
# print(committer_num_month)

past_six_months_file = 'past_six_months.txt'

# past_29_months_file = output_past_six_months(
#     n = 29, filename = 'past_29_months.txt'
# )

In [ ]:
past_29_months_file = 'past_29_months.txt'

with open(past_29_months_file, 'r') as past_29_txt:
    past_29_monthyears = [line.rstrip() for line in past_29_txt]

past_29_monthyears = pd.Series(past_29_monthyears)
past_29_num_months_arr = pd.to_datetime(past_29_monthyears).dt.strftime('%Y-%m').tolist()
# print(past_29_num_months_arr)

with open(past_six_months_file, 'r') as past_6_txt:
    past_6_monthyears = [line.rstrip() for line in past_6_txt]

# print(past_6_monthyears)

test_monthyear = past_6_monthyears[0]

# print(test_monthyear)
prev_test_monthyear = (parse(test_monthyear + ' 1') - relativedelta(months = 1)).strftime('%B %Y')
# print(prev_test_monthyear)

# test_6_monthyears = pd.Series([i - relativedelta(months = 1) for i in pd.to_datetime(past_6_monthyears)])
# test_6_monthyears = test_6_monthyears.dt.strftime('%B %Y').tolist()

# remember that the month selection can only be one month, which will be the sole element in the selection array
test_months_selection = past_6_monthyears[4]
test_months_selection = [test_months_selection] if type(test_months_selection) != list else test_months_selection
print(test_months_selection)

selection_for_mask = []
for selected_monthyear in test_months_selection:
    for offset in range(1, 2):
        monthyear_in_arr = (parse(selected_monthyear + ' 1') - relativedelta(months = offset)).strftime('%B %Y')
        if monthyear_in_arr not in selection_for_mask:
            selection_for_mask.append(monthyear_in_arr)
selection_for_mask = sorted(selection_for_mask, key = lambda s: pd.to_datetime(s))
print(selection_for_mask)

start_for_selection_for_mask_delta = selection_for_mask[0]
start_for_selection_for_mask_delta = [start_for_selection_for_mask_delta] if type(start_for_selection_for_mask_delta) != list else start_for_selection_for_mask_delta

selection_for_mask_delta = []
for start_monthyear in start_for_selection_for_mask_delta:
    for offset in range(1, 2):
        monthyear_for_delta_in_arr = (parse(start_monthyear + ' 1') - relativedelta(months = offset)).strftime('%B %Y')
        if monthyear_for_delta_in_arr not in selection_for_mask_delta:
            selection_for_mask_delta.append(monthyear_for_delta_in_arr)
selection_for_mask_delta = sorted(selection_for_mask_delta, key = lambda s: pd.to_datetime(s))
print(selection_for_mask_delta)

# official function to output the 12-month kpi and delta
# def prev_monthyears(monthyear_arr: list) -> list

mask_org = for_12_kpi_dfs['committer_monthyear'].isin(selection_for_mask)
raw_org_12_kpi_df = for_12_kpi_dfs.loc[mask_org].copy()
grouped_org_12_kpi_df = (
    raw_org_12_kpi_df['node_id'].count()
)
print(grouped_org_12_kpi_df) # output main kpi

mask_delta = for_12_kpi_dfs['committer_monthyear'].isin(selection_for_mask_delta)
raw_delta_12_kpi_df = for_12_kpi_dfs.loc[mask_delta].copy()
grouped_delta_12_kpi_df = (
    raw_delta_12_kpi_df['node_id'].count()
)
print(grouped_delta_12_kpi_df) # output number to calculate difference

percent_change_12_kpi = ((grouped_org_12_kpi_df - grouped_delta_12_kpi_df)/grouped_delta_12_kpi_df) * 100
print(float('{:.2f}'.format(percent_change_12_kpi))) # output 2nd kpi which is the % change

In [ ]:
with open(past_six_months_file, 'r') as past_six_months:
    past_six_monthyears = [line.rstrip() for line in past_six_months]
print(past_six_monthyears)

test_months_selection = past_six_monthyears[0]
test_months_selection = [test_months_selection] if type(test_months_selection) != list else test_months_selection
print(test_months_selection)

selection_for_mask = []
for selected_monthyear in test_months_selection:
    for offset in range(1, 7):
        monthyear_in_arr = (parse(selected_monthyear + ' 1') - relativedelta(months = offset)).strftime('%B %Y')
        if monthyear_in_arr not in selection_for_mask:
            selection_for_mask.append(monthyear_in_arr)
selection_for_mask = sorted(selection_for_mask, key = lambda s: pd.to_datetime(s))
print(selection_for_mask)

full_range = selection_for_mask + past_six_monthyears
print(full_range)

In [ ]:
import requests
import pprint

graphql_endpoint = 'https://api.github.com/graphql'

HEADERS = {
    'Authorization': 'Bearer ghp_xxx',
}

def run_query(query):
    request = requests.post(graphql_endpoint, json = {'query': query}, headers = HEADERS)
    if request.status_code == 200:
        return request.json()
    else:
        raise Exception(f'Query failed to run by returning code of {request.status_code, query}')

query = """
    query {
        __schema {
            types {
                name
                kind
                description
                fields {
                    name
                }
            }
        }
    }
"""

result = run_query(query) # execute the query
pprint.pprint(result)
objects_arr = result['data']['__schema']['types']
all_github_objects = []

for obj in objects_arr:
    if obj['fields'] != None:
        base_string = 'name: ' + obj['name'] + '\n' + 'description: ' + obj['description']
        if obj['fields'] == None:
            all_github_objects.append(base_string)
        if obj['fields'] != None:
            temp_fieldnames = [field['name'] for field in obj['fields']]
            temp_fieldnames_string = '\n'.join(temp_fieldnames)
            all_github_objects.append(base_string + '\n' + 'fields:' + '\n' + temp_fieldnames_string)

for string in all_github_objects:
    if 'additions' in string or 'deletions' in string:
        # print(string)
        # print(' ')
        pass

# objects containing additions and deletions
# Commit
# contains additions and deletions
# PullRequest
# contains additions and deletions
# PullRequestChangedFile
# contains additions and deletions